In [ ]:
# Library imports and setup
import os, sys
import numpy as np
from scipy import signal
from matplotlib import pyplot as plt
from dataclasses import dataclass
import multiprocessing as mp
import pandas as pd
import datetime as dt
import tqdm

# Use pop-up plots
# plt.switch_backend('TkAgg')

# Add the src folder to Python path
sys.path.append(os.path.abspath("../src"))
from utils import parse_timestamp, ensure_dir

In [ ]:
# Configuration

config = {
    'sdr_sample_rate': 2000000,
    'sdr_save_path': '../disk/2026-05-22_100ms/raw',
    'sdr_packet_path': '../disk/2026-05-22_100ms/packets',
    'sdr_ampl_threshold': 100, # amplitude threshold for the signal detection in ADC counts

    'ble_log_file': '../disk/2026-05-22_100ms/ble_log.csv'
}

In [ ]:
@dataclass
class Packet:
    preamble: bytes = None
    access_address: bytes = None
    pdu_header: bytes = None
    pdu_data: bytes = None
    file:str = None
    start_idx:int = None
    end_idx:int = None

In [ ]:
class ble_decoder:
    channel = 37
    data = []

    # Initialize LFSR with channel index (6 bits) + 7th bit as 1
    lfsr = 0x40 | (channel & 0x3F)

    def print_bits(self, data, end='\n'):
        """ Prints the values of the input iterable as integers without whitespace to enable the display of raw data bits.

        Args:
            data (iterable): Bits to be printed, either 1 or 0.
            end (str, optional): String to append to the end of the output. Defaults to '\n'.
        """
        print(str(list(map(int, data))).replace(' ', '').replace('\n', '').replace('[','').replace(']','').replace(',', ''), end=end)

    def print_bytes(self, data, sep='', end='\n'):
        """ Prints the values of the input iterable with the format :02x and separated by sep to enable easy display of byte values.

        Args:
            data (iterable): Bytes in the range 0...255.
            sep (str, optional): Separator used between the bytes. Defaults to ''.
            end (str, optional): String to append to the end of the output. Defaults to '\n'.
        """
        if data == None:
            print('', end=end)
            return
        
        print(sep.join(f"{byte:02x}" for byte in data), end=end)

    def whiten_bits(self, data):
        """ Whitenes the input data using the BLE 7-bit LFSR, starting at the first input entry (data[0]).
        The LFSR is persistent between runs and can be reset using the reset_lfsr method.

        Args:
            data (iterable): Bits to be whitened, either 1 or 0.

        Returns:
            np.array: Whitened bits.
        """
        result = []
        for bit in data:
            lsb = self.lfsr & 1
            self.lfsr >>= 1
            if lsb:
                self.lfsr ^= 0b01000100

            result.append(bit ^ lsb)
            # print(lsb, end='')
        # print('')
        return np.array(result, dtype=int)
    
    def reset_lfsr(self, channel = 37):
        """Resets the LFSR to the start value of 0x40 | (channel & 0x3F).

        Args:
            channel (int, optional): The BLE channel index. Defaults to 37.
        """
        self.channel = channel
        self.lfsr = 0x40 | (channel & 0x3F)

    def get_bytes (self, data, whiten = False):
        """Get bytes from an iterable of bits.
        Bit order is LSB first, byte order remains unchanged.
        The number of input bits must be a multiple of 8.

        Args:
            data (iterable): List of bit values, either 1 or 0.
            whiten (bool, optional): Wether to whiten the data using the LFSR or not. Defaults to False.

        Returns:
            bytes: Extracted bytes or None if the input length is not a multiple of 8.
        """
        if len(data) % 8:
            return None
        if whiten:
            data = self.whiten_bits(data)
        weights = np.tile([1,2,4,8,16,32,64,128], int(len(data)/8))
        data = weights * data[::] # weight the bits with their value
        data = np.reshape(data, (-1, 8)) # reshape for summing
        data = np.sum(data, 1) # sum each byte
        return bytes(list(data))

    def decode (self, data:np.array, thresh:float):
        """Decodes input IQ data into packets of bits.
        In doing so performs CFO compensation.
        TODO: Clock recovery could be implemented here.

        Args:
            data (np.array): IQ data samples.
            thresh (float): The amplitude threshold used to separate packets.

        Returns:
            tuple: Packet data as an np.array of ints, start indices as a list of ints and end indices as a list of ints.
        """
        if len(data) % 2 == 0:
            data = data[:-1]
        
        starts,ends = self.find_packets(data, thresh)

        if len(starts) == 0 or len(ends) == 0:
            return [], [], []

        data = np.array(np.abs(data) > thresh, dtype=int) * data
        data = np.unwrap(np.angle(data))

        if ends[0] < starts[0]:
            ends = ends[1:]
        if len(ends) < len(starts):
            starts = starts[:-1]
        
        for start, end in zip(starts,ends):
            data[start:end] = data[start:end] - np.linspace(data[start], data[end], end-start) # CFO compensation
            data[end:] -= data[end]

        data = np.diff(data)

        correlation = signal.correlate(
            np.array(data>0, dtype=int) * 2 - 1, 
            np.repeat(np.tile([1,-1], 4), 2), 
            mode='same'
        )
        starts = np.array((correlation >= 16).nonzero()[0])-10

        packets = []
        start_idx = []
        end_idx = []
        last_end = 0
        for start in starts:
            if start < last_end: # also "catches" index < 0
                continue
            for end in ends:
                if end > start:
                    end -= (end - start) % 8
                    packets.append(np.array(data[start:end:2] + data[start+1:end:2] > 0, dtype=int))
                    start_idx.append(start)
                    end_idx.append(end)
                    last_end = end
                    break

        return packets, start_idx, end_idx
    
        # # TODO: WIP: Clock recovery
        # interpolation = 4
        # data = signal.resample_poly(data, interpolation, 1)

        # idx = last_idx = 0
        # parsed_data = [0]
        # output_data = []
        # output_starts = []
        # sampling_pts = []
        # clk_error = [0]
        # while idx < len(data) - interpolation/2:
        #     parsed_data.append(1 if data[idx] > 0 else -1)
        #     output_data.append(1 if data[int(idx+interpolation/2)] > 0 else 0)
        #     sampling_pts.append(idx)

        #     clk_error.append((parsed_data[-1] * data[idx - 1] - parsed_data[-2] * data[idx] + 2*clk_error[-1])/4)
        #     if clk_error[-1] > 0.2: # TODO: Magic Number
        #         idx += 1
        #         #print(f'+1 bit at {idx}')
        #     elif clk_error[-1] < -0.2: # TODO: Magic Number
        #         idx -= 1
        #         #print(f'-1 bit at {idx}')
        #     idx += 2 * interpolation

        #     if set(range(last_idx, idx)) & set(starts*interpolation) != set():
        #         output_starts.append(len(output_data))
        #     last_idx = idx
        
        # plt.plot(data[(starts[0])*interpolation:starts[0]*interpolation+400])
        # plt.plot(np.repeat(clk_error[output_starts[0]:output_starts[0]+int(400/interpolation/2)], 8))
        # plt.eventplot(np.array(sampling_pts[output_starts[0]:output_starts[0]+int(400/interpolation/2)])-sampling_pts[output_starts[0]], lineoffsets=0, linelengths=2, colors="green")
        # plt.show()

        # return (output_data, output_starts)

        # for packet in packets:
        #     self.print_bits(packet)
    
    def find_packets (self, data:np.array, thresh):
        data = np.array(np.abs(data)) > thresh
        starts = np.nonzero(np.logical_and(data[1:], np.logical_not(data[:-1])))[0]
        ends =   np.nonzero(np.logical_and(np.logical_not(data[1:]), data[:-1]))[0]
        return (starts, ends)

In [ ]:
# Use the decoder class to decode the packets captured in the bursts on disk.

# Get the files from disk
files = sorted([f for f in os.listdir(config['sdr_save_path']) if f.endswith('.npy')])

# List to store the packet objects in
packets = []
# Decoder object used to decode and print the data
decoder = ble_decoder()

num_files = len(files)
num_files_decoded = 0

def parseFile (file):
    data = np.load(config['sdr_save_path']+'/'+file)

    # CFO compensation and decoding into bits
    data_chunks, starts, ends = decoder.decode(data, config['sdr_ampl_threshold'])

    packets = []

    # Parse the data chunks into Packet objects
    for data_chunk, start, end in zip(data_chunks, starts, ends):
        currentIndex = 0
        currentPacket = Packet()
        # General info
        currentPacket.file = file
        currentPacket.start_idx = start
        currentPacket.end_idx = end
        # Preamble
        currentPacket.preamble = decoder.get_bytes(data_chunk[currentIndex:currentIndex + 1*8]) # 1 byte @ 8 bits
        currentIndex += 1*8
        if currentPacket.preamble == None:
            continue # Drop packet
        # Access Address
        currentPacket.access_address = decoder.get_bytes(data_chunk[currentIndex:currentIndex + 4*8]) # 4 bytes @ 8 bits
        currentIndex += 4*8
        if currentPacket.access_address == None:
            continue # Drop packet
        currentPacket.access_address = currentPacket.access_address[::-1]
        # PDU
        decoder.reset_lfsr()
        currentPacket.pdu_header = decoder.get_bytes(data_chunk[currentIndex:currentIndex + 2*8], True) # 2 bytes @ 8 bits
        currentIndex += 2*8
        if currentPacket.pdu_header == None:
            continue # Drop packet
        if len(currentPacket.pdu_header) < 2:
            continue # Drop packet
        data_length = currentPacket.pdu_header[1]
        currentPacket.pdu_data = decoder.get_bytes(data_chunk[currentIndex:currentIndex + data_length*8], True) # data_length bytes @ 8 bits
        currentIndex += data_length*8
        if currentPacket.pdu_data == None:
            continue # Drop packet

        packets.append(currentPacket)
        
    return packets
        
# Essentially "for file in files:" using multiprocessing.
with mp.Pool(processes=mp.cpu_count()*2) as pool:
    packets = list(tqdm.tqdm(pool.imap(parseFile, files), total=len(files), desc='Decoding packets', unit='file'))

packets = [item for sublist in packets for item in sublist if item != None]

In [ ]:
# Print the decoded packets
for packet, nr in zip(packets, range(1, len(packets)+1)):
    print(f'Packet {nr}')
    print('Preamble:              ', end='')
    decoder.print_bytes(packet.preamble)
    print('Access Address:        ', end='')
    decoder.print_bytes(packet.access_address)
    print('PDU Header:            ', end='')
    decoder.print_bytes(packet.pdu_header)
    if packet.pdu_header != None:
        if (packet.pdu_header[0]&0x0F) in [0x0, 0x1, 0x2, 0x6]:
            print('Device Address:        ', end='')
            decoder.print_bytes(packet.pdu_data[:6][::-1], sep=':')
            print('Data:                  ', end='')
            decoder.print_bytes(packet.pdu_data[6:])
        else:
            print('PDU Data:              ', end='')
            decoder.print_bytes(packet.pdu_data)
    else:
        print('PDU Data:              ')
    
    print(f'File:                  {packet.file}')
    print(f'Start Index:           {packet.start_idx}')
    print(f'End Index:             {packet.end_idx}')

In [ ]:

# Use the decoded info to save the packets for the training data set.
ensure_dir(config['sdr_packet_path'])
for packet in tqdm.tqdm(packets, total=len(packets), desc='Storing labeled packets', unit='packet'):
    if (packet.pdu_header[0]&0x0F) in [0x0, 0x1, 0x2, 0x6]: # If the PDU type includes an address field
        if packet.pdu_data != None: # If the data was decoded successfully
            if len(packet.pdu_data[:6]) == 6: # Only include full adresses
                timestamp = parse_timestamp(packet.file[6:-4])
                timestamp += dt.timedelta(seconds=packet.start_idx/(packet.start_idx - packet.end_idx)/config['sdr_sample_rate'])
                filename = 'packet_' + ':'.join(f'{byte:02X}' for byte in packet.pdu_data[:6][::-1]) + '_' + str(timestamp)
                np.save(config['sdr_packet_path']+'/'+filename,np.load(config['sdr_save_path']+'/'+packet.file)[packet.start_idx:packet.end_idx])

In [ ]:
# Parse advertisements from the BLE log
# TODO: Use the parsed data
advertisements = []
with open(config['ble_log_file'], "r") as f:
    f.readline()
    for line in f:
        time,mac,rssi = line.replace('\n','').split(',')
        advertisements.append([parse_timestamp(time),mac,rssi])
advertisements = pd.DataFrame(advertisements, columns=['timestamp', 'mac', 'rssi'])
histogram = advertisements['mac'].value_counts()
plt.bar(histogram.index, histogram)
plt.xticks(rotation=90)
plt.show()